## Teoría base (cómo leer los bloques de código)

LangChain organiza el agente en componentes (prompt, modelo, tools, parser).

Bucle práctico:
1. interpretar objetivo,
2. elegir acción,
3. ejecutar tool,
4. observar y responder.

En cada bloque importante, revisa qué parte del bucle se implementa y cómo se valida el resultado.

## Guía de lectura (teoría ↔ práctica)

LangChain estructura el agente como una cadena de componentes (modelo, prompt, tools, parser, memoria opcional).

Bucle de agente (simplificado):
1. interpretar objetivo,
2. elegir acción/tool,
3. ejecutar,
4. observar,
5. iterar hasta respuesta final.

**Qué observar**: trazabilidad del razonamiento y robustez frente a preguntas ambiguas.

# LangChain + Ollama local

Vamos a hacer el mismo ejemplo de la calculadora pero usando LangChain y Ollama.

In [12]:
from langchain.tools import tool

@tool
def add(a: float, b: float) -> float:
    """
    Adds two numbers together.
    
    Args:
        a (float): The first number.
        b (float): The second number.
    """
    return a + b

@tool
def subtract(a: float, b: float) -> float:
    """
    Subtracts the second number from the first.
    
    Args:
        a (float): The first number.
        b (float): The second number.
    """
    return a - b

@tool
def multiply(a: float, b: float) -> float:
    """
    Multiplies two numbers together.
    
    Args:
        a (float): The first number.
        b (float): The second number.
    """
    return a * b

@tool
def divide(a: float, b: float) -> float | str:
    """
    Divides the first number by the second. Returns an error message if division by zero is attempted.
    
    Args:
        a (float): The first number.
        b (float): The second number.
    """
    if b == 0:
        return "Error: Division by zero is not allowed."
    return a / b

In [2]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent

llm = ChatOllama(model="qwen3:8b",
                 temperature=0.2)


agent = create_agent(llm, tools=[add, subtract, multiply, divide])

In [6]:
from pprint import pprint

for event in agent.stream(
    {"messages": [{"role": "user", "content": "What is 123 multiplied by 234, then subtract 12 and finally divide by 21?"}]},
):
    print("=== Event ===")
    pprint(event)

=== Event ===
{'model': {'messages': [AIMessage(content='123\u202f×\u202f234 = 28,782  \n28,782\u202f−\u202f12 = **28,770**  \n28,770\u202f÷\u202f21 = **1,370**', additional_kwargs={}, response_metadata={'model': 'gpt-oss:20b', 'created_at': '2026-03-05T16:34:14.476481051Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1697216027, 'load_duration': None, 'prompt_eval_count': 320, 'prompt_eval_duration': None, 'eval_count': 270, 'eval_duration': None, 'logprobs': None, 'model_name': 'gpt-oss:20b', 'model_provider': 'ollama'}, id='lc_run--019cbed9-8318-70c0-b79f-5763ee7d6328-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 320, 'output_tokens': 270, 'total_tokens': 590})]}}


# LangChain + Ollama Cloud

In [ ]:
import os
from langchain_ollama import ChatOllama

os.environ["OLLAMA_API_KEY"] = "<token>"


llm_ollama_cloud = ChatOllama(
    model="gpt-oss:20b-cloud",
    base_url="https://ollama.com",
    headers={"Authorization": f"Bearer {os.getenv('OLLAMA_API_KEY')}"}
)


In [5]:
from pprint import pprint
from langchain.agents import create_agent

agent = create_agent(llm_ollama_cloud, tools=[add, subtract, multiply, divide])
response = agent.invoke(
                {"messages": [{"role": "user", "content": "What is 15 multiplied by 3, then subtract 5 and finally divide by 2?"}]},
            )
pprint(response['messages'])

[HumanMessage(content='What is 15 multiplied by 3, then subtract 5 and finally divide by 2?', additional_kwargs={}, response_metadata={}, id='d33e1273-5ae8-4d5a-8b33-349391272539'),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gpt-oss:20b', 'created_at': '2026-03-05T16:33:55.416657844Z', 'done': True, 'done_reason': 'stop', 'total_duration': 712573576, 'load_duration': None, 'prompt_eval_count': 320, 'prompt_eval_duration': None, 'eval_count': 127, 'eval_duration': None, 'logprobs': None, 'model_name': 'gpt-oss:20b', 'model_provider': 'ollama'}, id='lc_run--019cbed9-00bc-7fc1-a026-f65b5175b994-0', tool_calls=[{'name': 'multiply', 'args': {'a': 15, 'b': 3}, 'id': '3d80e329-ffd0-4503-bded-ff00c831bea9', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 320, 'output_tokens': 127, 'total_tokens': 447}),
 ToolMessage(content='45.0', name='multiply', id='bf50eb47-fde3-45ee-adb2-c932317348c9', tool_call_id='3d80e329-ffd0-4503-bded-ff00c

# LangChain + Gemini

In [ ]:
import os
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

os.environ["GOOGLE_API_KEY"] = "<token>"

llm_gem = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3
)

agent = create_agent(llm_gem, tools=[add, subtract, multiply, divide])
response = agent.invoke(
                {"messages": [{"role": "user", "content": "What is 15 multiplied by 3, then subtract 5 and finally divide by 2?"}]},
            )
pprint(response['messages'])

In [ ]:
for event in agent.stream(
    {"messages": [{"role": "user", "content": "What is 123 multiplied by 234, then subtract 12 and finally divide by 21?"}]},
):
    print("=== Event ===")
    pprint(event)

# LangChain + OpenAI API

In [ ]:
# !pip install -U langchain-openai

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "<token>"

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from pprint import pprint

llm = ChatOpenAI(
    model="gpt-4o",
)

agent = create_agent(llm, tools=[add, subtract, multiply, divide])

response = agent.invoke(
                {"messages": [{"role": "user", "content": "What is 15 multiplied by 3, then subtract 5 and finally divide by 2?"}]},
            )
pprint(response['messages'])

# Ejercicio

Implementar el agent SQL con LangChain.

In [3]:
import os
import sqlite3
from pathlib import Path
from pprint import pprint
from typing import Any, Dict

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_ollama import ChatOllama

SQLITE_DB_PATH = Path("Chinook_Sqlite.sqlite")
if not SQLITE_DB_PATH.exists():
    SQLITE_DB_PATH = Path("dia4-2") / "Chinook_Sqlite.sqlite"

def get_connection():
    """Return connection to the sqlite database."""
    conn = sqlite3.connect(str(SQLITE_DB_PATH))
    conn.row_factory = sqlite3.Row
    return conn

def _query_database_impl(query: str) -> Dict[str, Any]:
    """Internal helper to run read-only SQL queries."""
    sql = query.strip().rstrip(";")
    if not sql.lower().startswith("select"):
        return {"ok": False, "error": "Only SELECT queries are allowed."}

    try:
        with get_connection() as conn:
            cursor = conn.execute(sql)
            rows = cursor.fetchmany(100)
            columns = [description[0] for description in cursor.description] if cursor.description else []

        result_rows = [dict(row) for row in rows]
        return {
            "ok": True,
            "columns": columns,
            "rows": result_rows,
            "row_count": len(result_rows),
            "truncated": len(result_rows) == 100,
        }
    except Exception as error:
        return {"ok": False, "error": str(error)}

@tool
def query_database(query: str) -> Dict[str, Any]:
    """Run a SELECT SQL query against the database and return results.

    Args:
        query: SELECT SQL query to execute.

    Returns:
        A dictionary with query results (columns, rows, etc.).
    """
    return _query_database_impl(query)

@tool
def show_tables() -> Dict[str, Any]:
    """Return the list of database tables."""
    result = _query_database_impl(
        "SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name"
    )
    if not result.get("ok"):
        return result

    table_names = [row["name"] for row in result["rows"]]
    return {"ok": True, "tables": table_names, "table_count": len(table_names)}

@tool
def describe_table(table_name: str) -> Dict[str, Any]:
    """Return the schema of the specified table.

    Args:
        table_name: Name of the table to describe.

    Returns:
        A dictionary with the schema information.
    """
    cleaned_name = table_name.strip().replace("'", "''")
    sql = f"PRAGMA table_info('{cleaned_name}')"

    try:
        with get_connection() as conn:
            cursor = conn.execute(sql)
            rows = cursor.fetchall()

        columns = [dict(row) for row in rows]
        if not columns:
            return {"ok": False, "error": f"Table '{table_name}' not found."}

        return {"ok": True, "table": table_name, "columns": columns}
    except Exception as error:
        return {"ok": False, "error": str(error)}

ollama_api_key = os.getenv("OLLAMA_API_KEY")
if not ollama_api_key:
    raise ValueError("Define OLLAMA_API_KEY antes de ejecutar esta celda.")

llm_ollama_cloud = ChatOllama(
    model="gpt-oss:20b-cloud",
    base_url="https://ollama.com",
    temperature=0.2,
    headers={"Authorization": f"Bearer {ollama_api_key}"},
)

agent_sql = create_agent(
    llm_ollama_cloud,
    tools=[query_database, show_tables, describe_table],
)

response = agent_sql.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Who is the customer that spent the most money? Give me the name",
            }
        ]
    }
)

pprint(response["messages"])

[HumanMessage(content='Who is the customer that spent the most money? Give me the name', additional_kwargs={}, response_metadata={}, id='5a67ce90-99c9-4971-a121-5bcbc1c755a2'),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gpt-oss:20b', 'created_at': '2026-03-05T16:44:31.676527527Z', 'done': True, 'done_reason': 'stop', 'total_duration': 431215156, 'load_duration': None, 'prompt_eval_count': 233, 'prompt_eval_duration': None, 'eval_count': 30, 'eval_duration': None, 'logprobs': None, 'model_name': 'gpt-oss:20b', 'model_provider': 'ollama'}, id='lc_run--019cbee2-f3fc-76c0-a9c9-df864b6bcf3d-0', tool_calls=[{'name': 'show_tables', 'args': {}, 'id': '40134a40-c2ad-4fdd-a72b-900d3beac373', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 233, 'output_tokens': 30, 'total_tokens': 263}),
 ToolMessage(content='{"ok": true, "tables": ["Album", "Artist", "Customer", "Employee", "Genre", "Invoice", "InvoiceLine", "MediaType", "Playlist", "P